<!-- cabecera-entorno -->
## Antes de empezar

**Clase 14 · Introducción a Machine Learning** — Bloque 2 · Demo. Este cuaderno se recorre **por su
cuenta**: explica cada concepto antes de usarlo, y el profesor circula por el salón resolviendo
dudas. No hay que esperar a que alguien lo dicte.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import sklearn
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

print("scikit-learn:", sklearn.__version__)

# Los numeros de este cuaderno (R2 de prueba, gaps, importancias) se calcularon con
# scikit-learn 1.9.0. Con una version anterior el arbol puede partir distinto y las
# cifras dejan de coincidir con las de las notas y las del companero de al lado.
_version_sklearn = tuple(int(parte) for parte in sklearn.__version__.split(".")[:2])
if _version_sklearn < (1, 9):
    print("AVISO: este cuaderno se calculo con scikit-learn 1.9.0 o superior.",
          "Con una version anterior sus numeros pueden diferir de los del material.",
          "Actualice con: pip install -r ../requirements.txt")

RUTA_VERIFICACION = "../datos/HISTORICO_CONSUMO.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 14 · Demo — Entrenar, evaluar e interpretar un modelo

**Dataset:** `../datos/HISTORICO_CONSUMO.csv` (consumo de agua de Empocaldas, el mismo de las
clases 1, 4 y 5)

## Cómo se usa este cuaderno

Usted avanza solo, leyendo. Cada bloque de código viene precedido de la explicación del concepto que
usa, y cada término nuevo se define la primera vez que aparece. El profesor circula por el salón: si
algo no cierra, levante la mano en el momento, no al final.

**El recorrido, de lo simple a lo complejo:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 | Los datos, y qué tienen de sucio | Un DataFrame en el que se puede confiar |
| 1 | Qué es entrenar un modelo, de verdad | La diferencia entre programar una regla y deducirla |
| 2 | Qué se predice, con qué, y qué es una fuga de datos | `X` y `y` bien armados |
| 3 | Por qué se parte en entrenamiento y prueba | La única evaluación que significa algo |
| 4 | El patrón de cinco pasos de scikit-learn | Un modelo entrenado y evaluado |
| 5 | Sobreajuste: verlo en una tabla, no creerlo | La regla del gap |
| 6 | Qué mide cada métrica, y por qué una sola engaña | Un reporte honesto |
| 7 | Interpretar el modelo sin inventar causalidad | Una frase que se pueda decir en una reunión |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Escribir código es el bloque 3, con el reto, y es
lo que se entrega.

**Las catorce preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.** Abrirlo
antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación del Momento 3 es que usted
sepa mirar el resultado de un modelo y decir qué significa, no que sepa reproducir un cálculo.

**Las cajas "Para entender qué está pasando"** van en bloque citado, con la barra vertical a la
izquierda. Explican el mecanismo de Python o de scikit-learn que hay por debajo: qué es un objeto con
estado, qué significa el guion bajo final de `feature_importances_`, qué es una semilla. **No son
materia de analítica, son el piso para entenderla.** Si ya lo sabe, sáltelas sin culpa.

**Lo que ya se explicó no se repite.** La limpieza de tipos es de la clase 3, `describe()` y los
outliers son de la clase 4, correlación y causalidad es de la clase 5, y el muestreo es de la clase
13. Aquí se enlaza y se sigue; lo que se desarrolla completo es lo nuevo.

**Si algo se rompe**, no siempre es un accidente: hay tres errores provocados a propósito, con
`try / except` y la explicación al lado. Son los tres que más se ven el día del proyecto final.

---

## 0. Los datos, antes de tocarlos

Es el consumo de agua de Empocaldas, el mismo de las clases 1, 4 y 5. **Eso es a propósito:** hoy lo
nuevo es el modelo, no el dominio. Si tuviéramos que aprender un dataset nuevo, media clase se iría en
entender qué es un m3.

| Campo | Valor |
|-------|-------|
| Fuente | Empresa de Obras Sanitarias de Caldas (Empocaldas), datos.gov.co |
| Grano | Un registro por municipio, año, mes y estrato |
| Tamaño | 21.816 filas, 12 columnas |
| Cobertura | 2015 a 2023 (2023 está incompleto) |
| Municipios | 24 valores distintos: municipios de Caldas y corregimientos que se facturan aparte |
| Estratos | Estrato1 a Estrato6, Industrial, Comercial, Publico / Oficial |

| Columna | Qué mide |
|---------|----------|
| `AÑO`, `MES` | Periodo del registro |
| `MUNICIPIO` | Municipio de Caldas |
| `ESTRATO` | Tipo de suscriptor |
| `No. SUSCRIPTORES ACUEDUCTO` | Cuántos suscriptores hay en esa combinación |
| `CONSUMO M3 ACUEDUCTO` | Consumo total del grupo, en m3 |
| `PROMEDIO CONSUMO ACUEDUCTO` | Consumo por suscriptor, en m3. **Esto es lo que vamos a predecir** |

Las tres columnas de alcantarillado son análogas. `NIT` y `RAZON SOCIAL` son constantes: una sola
empresa, ninguna información.

**Lo que está sucio** ya lo conoce de la clase 3, así que aquí no se vuelve a explicar la técnica, solo
se aplica: `AÑO` es texto con coma de miles (`"2,015"`), los números mezclan convenciones (`"1.043"`
son mil cuarenta y tres, pero `"3.02"` son tres coma cero dos), hay nulos, y hay valores imposibles.

Lo único que sí se desarrolla completo, porque es nuevo, es **cómo se decide una regla de limpieza
cuando el archivo es ambiguo, y cómo se verifica esa decisión**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

plt.rcParams["figure.figsize"] = (9, 5)

# La semilla del curso. Aparece en cada train_test_split y en cada arbol, y por eso
# sus resultados van a coincidir con los del compañero de al lado y con los de la
# proxima vez que ejecute el cuaderno. El numero en si no tiene nada de especial.
SEMILLA = 42

RUTA = "../datos/HISTORICO_CONSUMO.csv"

# dtype=str a proposito: hoy queremos ver el texto crudo antes de convertir nada.
crudo = pd.read_csv(RUTA, dtype=str)

print("Forma:", crudo.shape)
crudo.head(4)

**La radiografía del archivo, antes de decidir nada.** Son las funciones de la clase 2 y aquí no son
un repaso: cada una responde una pregunta que hay que tener contestada antes de elegir qué entra al
modelo.

| Función | La pregunta que responde | Por qué importa hoy |
|---------|--------------------------|---------------------|
| `crudo.shape` | ¿Cuántas filas y columnas hay? | Con pocas filas no hay cómo apartar un examen honesto |
| `crudo.head()` / `crudo.tail()` | ¿Cómo se ven el principio y el final? | El final del archivo es donde suelen esconderse los totales y las filas basura |
| `crudo.info()` | ¿Qué columnas hay, de qué tipo y cuántas no nulas? | Aquí todo entró como texto a propósito: el modelo solo entiende números |
| `crudo.dtypes` | ¿De qué tipo quedó cada columna? | Es la comprobación de que la limpieza de la sección 0.2 hizo lo suyo |
| `value_counts()` | ¿Cómo se reparten las categorías? | `ESTRATO` se va a convertir en columnas con `get_dummies`: cada categoría es una columna |
| `nunique()` | ¿Cuántos valores distintos hay? | Dice de antemano cuántas columnas va a agregar `get_dummies` |

In [ ]:
crudo.info()

In [ ]:
print("Ultimas 3 filas, para descartar totales o basura al final del archivo:")
print(crudo.tail(3).to_string(index=False))

print("\nTipos con los que entro cada columna (todo texto, a proposito):")
print(crudo.dtypes.value_counts().to_string())

print("\nReparto de ESTRATO, que va a convertirse en columnas del modelo:")
print(crudo["ESTRATO"].value_counts().to_string())

print("\nValores distintos por columna categorica:")
print(crudo[["MUNICIPIO", "ESTRATO", "AÑO", "MES"]].nunique().to_string())

### 0.1 Mire la suciedad con sus propios ojos

Limpiar sin haber mirado es aplicar una receta. La celda de abajo imprime el texto crudo de las
columnas que vamos a usar.

In [ ]:
print("Valores unicos de AÑO (texto, con coma de miles):")
print(crudo["AÑO"].unique())

print("\nSuscriptores acueducto - primeros valores distintos:")
print(crudo["No. SUSCRIPTORES ACUEDUCTO"].dropna().unique()[:10].tolist())

print("\nPromedio de consumo - ejemplos con coma (fijese en '6,757.6'):")
ejemplos = crudo["PROMEDIO CONSUMO ACUEDUCTO"].dropna()
print(ejemplos[ejemplos.str.contains(",")].unique()[:5].tolist())

print("\nNulos por columna:")
print(crudo.isna().sum()[crudo.isna().sum() > 0].to_string())

### 0.2 La regla de limpieza que decidimos nosotros

El archivo usa el punto para dos cosas distintas:

- `"1.043"` es mil cuarenta y tres. El punto es **separador de miles**.
- `"3.02"` es tres coma cero dos. El punto es **decimal**.

Ningún `astype(float)` resuelve eso solo, porque la ambigüedad no está en el código: está en el
archivo. Hay que **decidir** una regla y hacerse responsable de ella. La nuestra:

> El punto es separador de miles **solo si le siguen exactamente tres dígitos**. Cualquier otro punto
> es decimal.

En código, `str.replace(r"\.(?=\d{3}\b)", "", regex=True)`.

> **Para entender qué está pasando: qué es ese `(?=...)`**
>
> Es una *expresión regular* (regex), el lenguaje de patrones de texto que vio en la clase 3. El
> `(?=\d{3}\b)` se llama **lookahead**: significa "seguido de tres dígitos y fin de grupo", pero sin
> consumir esos dígitos, así que solo se borra el punto y los dígitos quedan. La `r` delante de las
> comillas es una *raw string*: le dice a Python que no interprete las barras invertidas, que aquí son
> parte del patrón.
>
> No hace falta memorizar la regex. Hace falta poder explicar la **regla** y por qué la eligió.

**Lo importante no es la regex, es lo que viene después:** una regla que uno decide se verifica. Si el
parseo es correcto, `consumo total / suscriptores` tiene que parecerse al promedio reportado. Eso es
lo que hace la sección 0.3.

In [ ]:
def a_numero(serie):
    """Convierte texto a numero respetando las dos convenciones del archivo.

    1. Quita la coma de miles.
    2. Quita el punto SOLO si le siguen exactamente 3 digitos (separador de miles).
    """
    limpia = serie.str.replace(",", "", regex=False)
    limpia = limpia.str.replace(r"\.(?=\d{3}\b)", "", regex=True)
    return pd.to_numeric(limpia, errors="coerce")


COLUMNAS_NUMERICAS = [
    "No. SUSCRIPTORES ACUEDUCTO",
    "CONSUMO M3 ACUEDUCTO",
    "PROMEDIO CONSUMO ACUEDUCTO",
]

df = crudo.copy()
for columna in COLUMNAS_NUMERICAS:
    df[columna] = a_numero(df[columna])

# El anio tambien viene con coma de miles: "2,015"
df["anio"] = pd.to_numeric(df["AÑO"].str.replace(",", "", regex=False))

# El mes viene como texto en mayusculas, y un modelo no sabe que ENERO va antes que FEBRERO
MESES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
         "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"]
df["mes_num"] = df["MES"].map({nombre: i + 1 for i, nombre in enumerate(MESES)})

print(df[["anio", "mes_num"] + COLUMNAS_NUMERICAS].describe().round(2).T.to_string())

### 0.3 Verificamos la regla

Si el parseo está bien, `CONSUMO M3 / SUSCRIPTORES` debería parecerse al `PROMEDIO` reportado. Este
chequeo cruzado es lo que separa una limpieza responsable de una a ciegas: no confirma que la regla
sea perfecta, pero sí detectaría que es un desastre.

In [ ]:
razon = df["CONSUMO M3 ACUEDUCTO"] / df["No. SUSCRIPTORES ACUEDUCTO"]
comparables = razon.notna() & df["PROMEDIO CONSUMO ACUEDUCTO"].notna()
coincide = np.abs(razon - df["PROMEDIO CONSUMO ACUEDUCTO"]) < 0.05 * df["PROMEDIO CONSUMO ACUEDUCTO"]

print(f"Filas comparables : {comparables.sum()}")
print(f"Coinciden (+-5%)  : {(coincide & comparables).sum()}")
print(f"Porcentaje        : {100 * (coincide & comparables).sum() / comparables.sum():.1f} %")
print()
print("Un 91% de coincidencia valida la regla del punto.")
print("El 9% restante son inconsistencias del archivo original, no de nuestra limpieza.")

### 0.4 Filtramos valores imposibles

El promedio de consumo llega a valores de más de 100.000 m3 **por suscriptor**. Una vivienda consume
del orden de 10 a 20 m3 al mes. Eso no es un outlier interesante de los de la clase 4 (la jirafa en
el parque de perros): es basura del archivo.

Filtramos a un rango plausible y **decimos cuántas filas perdimos**. Ocultar el filtro sería
deshonesto: quien lee el informe tiene derecho a saber sobre qué datos se entrenó el modelo.

In [ ]:
LIMITE_SUPERIOR = 100  # m3 por suscriptor al mes

antes = len(df)
modelo_df = df[
    (df["PROMEDIO CONSUMO ACUEDUCTO"] > 0)
    & (df["PROMEDIO CONSUMO ACUEDUCTO"] <= LIMITE_SUPERIOR)
    & (df["No. SUSCRIPTORES ACUEDUCTO"] > 0)
].copy()

print(f"Filas antes   : {antes}")
print(f"Filas despues : {len(modelo_df)}")
print(f"Descartadas   : {antes - len(modelo_df)}  ({100 * (antes - len(modelo_df)) / antes:.1f} %)")
print()
print(modelo_df["PROMEDIO CONSUMO ACUEDUCTO"].describe().round(2).to_string())

**Pregunta de interpretación 1.** El filtro descartó 6.347 de las 21.816 filas: el 29,1% del
archivo. ¿Qué tiene que aparecer en el informe por haber botado casi un tercio de los datos, y qué
riesgo introduce ese filtro sobre las conclusiones del modelo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Lo que va en el informe son tres cosas, no una:** cuántas filas se descartaron, con qué criterio
exacto (promedio mayor que 0 y menor o igual a 100 m3 por suscriptor, y suscriptores mayor que 0), y
por qué ese criterio y no otro. Un número solo —"se limpiaron los datos"— no es reportar, es tapar.

**El riesgo tiene nombre: el filtro no borra al azar.** Buena parte de lo que cae son filas con cero
suscriptores o cero consumo, que se concentran en municipios pequeños y en los primeros años del
archivo. Es decir, el modelo va a quedar entrenado sobre una población que no es exactamente la
original, y todo lo que diga vale para **esa** población. Si mañana alguien lo usa para estimar el
consumo de un corregimiento diminuto, está fuera del rango donde el modelo aprendió.

La forma honesta de decirlo en la sustentación: *"el modelo se entrenó sobre las 15.469 filas con
consumo por suscriptor entre 0 y 100 m3; las 6.347 restantes eran ceros o valores físicamente
imposibles, y su exclusión sesga el resultado hacia los grupos con operación regular"*. Esa frase se
sostiene. "Limpiamos los datos" no.

</details>

---

## 1. Qué es entrenar un modelo

Antes de escribir una línea de scikit-learn, hay que tener clara una idea, porque todo lo demás
depende de ella.

**El problema.** Empocaldas quiere estimar cuántos m3 va a consumir un suscriptor promedio en una zona
de la que todavía no tiene historia. Intente escribir el `if` que lo calcula:

```
si el estrato es 1        -> 10 m3
si el estrato es 1 y es diciembre -> ¿12?
si es La Dorada, que es caliente  -> ¿más?
si es un colegio y no una casa    -> ¿mucho más?
```

Cada respuesta agrega una condición, y ningún conjunto de condiciones termina de cubrir el caso. No
es que sea difícil: es que **la regla existe, está en los datos, pero usted no la puede escribir a
mano.**

**Machine learning es exactamente eso: lo que se hace cuando la regla existe y no se puede escribir.**
En vez de escribirla, se le muestran ejemplos al computador y él la deduce.

| Programación tradicional | Machine learning |
|--------------------------|------------------|
| Usted pone los **datos** y las **reglas** | Usted pone los **datos** y las **respuestas** |
| El computador produce las **respuestas** | El computador produce las **reglas** |

Es la misma caja al revés. Y de ahí salen dos consecuencias que no son obvias:

- **Lo que el modelo puede reconocer lo determinan los ejemplos que vio.** Si solo le muestra
  viviendas, va a fallar con un hospital.
- **Datos malos, modelo malo.** Ninguna cantidad de algoritmo arregla ejemplos mal etiquetados. Por eso
  la clase 3 no era un trámite previo.

**Entrenar** es, entonces, el proceso de encontrar esa regla a partir de ejemplos. Nada más. La palabra
suena a inteligencia y no lo es: es reconocimiento de patrones a escala.

En la sección 4 vamos a **ver la regla que el modelo dedujo**, escrita en texto. No es una metáfora.

---

## 2. Qué se predice, con qué, y la trampa

### 2.1 `y` es lo que quiere saber; `X` es lo que sabe antes de saberlo

**`y` (la variable objetivo):** `PROMEDIO CONSUMO ACUEDUCTO`, los m3 que consume un suscriptor
promedio en ese municipio, mes y estrato.

Es un **número** → es un problema de **regresión** → `DecisionTreeRegressor`.

> **Regla de bolsillo para todo el semestre:** mire el tipo de la columna que quiere predecir. Si es
> un número, regresión. Si es una categoría, clasificación. Nada más.

**Para qué le sirve a alguien:** si Empocaldas puede estimar el consumo por suscriptor de una zona
nueva, puede dimensionar la infraestructura y proyectar facturación sin esperar un año de historia.
Esa frase no es decoración: es la pregunta 1 de las cuatro que su proyecto final tiene que responder.

### 2.2 La trampa: fuga de datos

Ahora la pregunta que decide la calidad de todo el proyecto. **¿Por qué NO metemos
`CONSUMO M3 ACUEDUCTO` entre las variables de entrada?**

Porque:

```
PROMEDIO  ≈  CONSUMO M3  /  SUSCRIPTORES
```

Si le doy al modelo el consumo total **y** el número de suscriptores, le estoy dando la respuesta
partida en dos pedazos. Va a "acertar" con un R2 altísimo sin haber aprendido nada: el día que llegue
un barrio nuevo no va a existir el consumo total, porque **eso es justamente lo que se quería
predecir**.

Se llama **fuga de datos** (*data leakage*), y es el error más caro del proyecto final porque no
produce un error: produce un resultado espectacular.

> **Regla que hay que memorizar:** en `X` solo va información que va a tener **antes** de conocer `y`
> en el mundo real.

En la sección 6 lo ejecutamos con fuga, para que vea el número dispararse.

La celda de abajo deja escrito, y en una variable, cuál es la columna del archivo que **no** puede
entrar en `X`. No es una elección de estilo: es la decisión que define si el modelo mide algo o mide
su propia respuesta.

In [ ]:
columna_con_fuga = "CONSUMO M3 ACUEDUCTO"

print("No entra en X:", columna_con_fuga)
print("Porque", columna_con_fuga, "/ No. SUSCRIPTORES ACUEDUCTO es, aproximadamente,")
print("la columna que queremos predecir. Seria darle la respuesta al modelo.")

**Pregunta de interpretación 2.** `CONSUMO M3 ACUEDUCTO` no es la única columna sospechosa: las tres
de alcantarillado también están en el archivo. ¿Entrarían en `X` sin problema? Y en el dataset de su
proyecto final, ¿qué pregunta concreta le va a hacer a cada columna antes de dejarla entrar?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Las de alcantarillado son un caso más sutil, y por eso vale la pena.** No reproducen la respuesta
aritméticamente, así que no son fuga en el sentido estricto. Pero el consumo de alcantarillado se
factura casi siempre a partir del de acueducto: es información que, en el mundo real, **también
llega después**. Si el objetivo es estimar el consumo de una zona sin historia, ahí tampoco va a
haber consumo de alcantarillado. La regla no es "¿reproduce la respuesta?", es "¿voy a tener este
dato en el momento en que necesite la predicción?".

**La pregunta que hay que hacerle a cada columna** es exactamente esa, y se responde con el
calendario en la mano, no con el DataFrame: *¿este valor existe ANTES de que exista `y`?* Si la
respuesta es "existe al mismo tiempo" o "existe después", la columna se cae, por muy predictiva que
sea. De hecho, cuanto más predictiva, más sospechosa.

**El caso clásico en un proyecto de aula:** predecir si un estudiante desertó, y meter entre las
variables el número de materias matriculadas el semestre siguiente. Correlación perfecta, R2 de
ensueño, modelo inútil.

</details>

### 2.3 Categorías: `pd.get_dummies`

`ESTRATO` y `MUNICIPIO` son texto, y un árbol parte preguntando "¿esta variable es mayor que X?". Eso
no significa nada sobre texto: no hay un orden entre "Comercial" e "Industrial".

`pd.get_dummies` convierte cada categoría en su propia columna de 0 y 1, y entonces la pregunta pasa a
ser "¿es este municipio, sí o no?", que sí tiene sentido. `drop_first=True` elimina una de las
columnas resultantes porque es redundante: si todas las demás están en 0, ya sabe cuál era.

> **Para entender qué está pasando: por qué el modelo solo entiende números**
>
> Un modelo de scikit-learn es, por dentro, aritmética sobre una matriz de números. No hay ningún
> lugar donde quepa la palabra "Comercial". Por eso todo lo que entre en `X` tiene que ser numérico y
> sin faltantes, y por eso los dos errores más comunes del día son exactamente esos dos. Los vamos a
> provocar en un momento.

In [ ]:
COLUMNA_OBJETIVO = "PROMEDIO CONSUMO ACUEDUCTO"

COLUMNAS_ENTRADA = [
    "anio",
    "mes_num",
    "No. SUSCRIPTORES ACUEDUCTO",
    "ESTRATO",
    "MUNICIPIO",
]
# Nota: CONSUMO M3 ACUEDUCTO NO esta en esta lista. Seria fuga de datos.

datos_modelo = modelo_df[COLUMNAS_ENTRADA + [COLUMNA_OBJETIVO]].dropna()

X = pd.get_dummies(datos_modelo[COLUMNAS_ENTRADA],
                   columns=["ESTRATO", "MUNICIPIO"], drop_first=True)
y = datos_modelo[COLUMNA_OBJETIVO]

print(f"X: {X.shape[0]} filas, {X.shape[1]} columnas")
print(f"y: {y.shape[0]} valores, promedio {y.mean():.2f} m3")
print()
print("Primeras columnas de X:", X.columns[:5].tolist())
print("Ultimas columnas de X :", X.columns[-3:].tolist())

**Pregunta de interpretación 3.** `COLUMNAS_ENTRADA` tiene cinco nombres y `X` salió con **34
columnas**. ¿De dónde salieron las otras 29? ¿Y qué pasaría si en su proyecto una de las columnas
categóricas tuviera 500 valores distintos, como una cédula o un código de producto?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

De las cinco entradas, tres son numéricas y pasan tal cual (`anio`, `mes_num`, suscriptores). Las
otras dos son categóricas, y `get_dummies` le crea una columna a cada valor: `ESTRATO` tiene 9
valores y `MUNICIPIO` tiene **24**. Con `drop_first=True` se descarta uno de cada grupo, así que
quedan 8 + 23 = 31, más las 3 numéricas: 34. La cuenta cierra exacta.

Lo interesante es el 24. Caldas tiene 27 municipios y el archivo trae 24 valores, así que no es "un
valor por municipio": hay corregimientos que Empocaldas factura aparte (Guarinocito, Kilómetro 41,
Arma) y municipios donde no opera. **Eso solo se descubre mirando `X.columns` o
`crudo["MUNICIPIO"].nunique()`, no suponiendo.** Suponer categorías es la vía rápida a un `X` con
más columnas de las que uno cree.

**Con 500 categorías el método se rompe**, y de dos maneras a la vez. Primero, `X` pasaría a tener
500 columnas para 15.000 filas: cada columna sería casi toda ceros, y el árbol tendría 500 preguntas
posibles con poquísimos ejemplos para decidir cada una. Eso es una receta de sobreajuste. Segundo, y
más importante: un identificador **no es una categoría útil**. Una cédula no dice nada del mundo,
dice quién es la fila. Un modelo que aprende de cédulas memorizó personas, no patrones.

Qué se hace: agrupar (departamento en vez de municipio, familia de producto en vez de producto), o
reemplazar la categoría por algo que sí describe (población del municipio, precio medio de la
familia). La decisión se toma antes de `get_dummies`, no después.

</details>

### 2.4 Los dos errores que van a ver el día del proyecto

Provocados a propósito, con `try / except` para que el cuaderno siga corriendo. Léalos ahora con
calma: dentro de dos semanas, con la sustentación encima, va a reconocer el mensaje en dos segundos en
vez de en veinte minutos.

In [ ]:
# Error 1: pasar texto sin codificar
try:
    DecisionTreeRegressor(random_state=SEMILLA).fit(datos_modelo[COLUMNAS_ENTRADA], y)
except ValueError as error:
    print("Tipo de error:", type(error).__name__)
    print("Mensaje:", str(error).split("\n")[0])
    print("-> Falto pd.get_dummies sobre ESTRATO y MUNICIPIO.")

print()

# Error 2: dejar valores faltantes en X
X_con_nulos = X.copy()
X_con_nulos.loc[X_con_nulos.index[:5], "No. SUSCRIPTORES ACUEDUCTO"] = np.nan
try:
    DecisionTreeRegressor(random_state=SEMILLA).fit(X_con_nulos, y)
except ValueError as error:
    print("Tipo de error:", type(error).__name__)
    print("Mensaje:", str(error).split("\n")[0])
    print("-> Falto dropna() (o imputar) ANTES de entrenar. Cinco NaN bastan para tumbarlo.")

---

## 3. Por qué se parte en entrenamiento y prueba

**Sin esto, todo lo demás es ritual.** Es la sección que hay que entender aunque se olvide el código.

### La analogía del examen

| Estudiar | Machine learning |
|----------|------------------|
| El taller que resolvió en clase, con respuestas | Conjunto de **entrenamiento** (80%) |
| El examen final, que nunca vio | Conjunto de **prueba** (20%) |
| Estudiar el taller | `modelo.fit(X_train, y_train)` |
| Presentar el examen | `modelo.predict(X_test)` |
| Su nota real | La métrica sobre el conjunto de **prueba** |

Hay **dos formas de sacar 5.0 en el taller**: entender el tema, o memorizar las respuestas sin
entender nada. Miradas solo desde el taller, las dos se ven idénticas. La única manera de
distinguirlas es un examen con preguntas que no estaban en el taller.

Por eso el 20% se aparta **antes** de entrenar y no se toca hasta el final. Evaluar sobre los datos
con los que se entrenó es autocalificarse el taller: siempre da bien y no significa nada.

**Qué significa entonces "evaluar sobre datos que el modelo no vio":** medir si la regla que dedujo
sirve para casos nuevos, que es lo único que va a pasar cuando el modelo se use de verdad. Esa medida,
y solo esa, es la que se reporta.

| Argumento | Qué hace |
|-----------|----------|
| `X, y` | Se parten juntos, fila por fila. Nunca por separado, o las respuestas dejarían de corresponder |
| `test_size=0.2` | El 20% se aparta para el examen |
| `random_state=SEMILLA` | Fija la aleatoriedad para que la partición sea siempre la misma |

> **Para entender qué está pasando: qué es una semilla**
>
> `train_test_split` elige al azar qué filas van a cada lado. "Al azar" en un computador significa una
> secuencia de números que parece aleatoria pero que se calcula, y la **semilla** es el punto de
> partida de ese cálculo. Misma semilla, misma secuencia, misma partición. Sin semilla, cada ejecución
> parte distinto, sus números cambian y no puede comparar dos modelos ni reproducir el resultado del
> compañero. Por eso `random_state=SEMILLA` va en todo lo que tenga azar adentro.
>
> Esto no es lo mismo que el muestreo de la clase 13, pero se parece: allá se apartaba una muestra para
> estimar un parámetro de la población; aquí se aparta una parte de los datos para estimar el
> desempeño del modelo. En los dos casos, la parte apartada tiene que ser representativa y no estar
> contaminada.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)

print(f"Entrenamiento: {len(X_train)} filas  ({100 * len(X_train) / len(X):.0f} %)")
print(f"Prueba       : {len(X_test)} filas  ({100 * len(X_test) / len(X):.0f} %)")
print()
print("Esas filas de prueba el modelo NO las va a ver hasta el final.")

**Pregunta de interpretación 4.** ¿Por qué 80/20 y no 50/50? ¿Y qué haría si su dataset tuviera 100
filas en total?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Es un balance entre dos cosas que se pelean. Con más datos de entrenamiento el modelo aprende mejor;
con más datos de prueba la evaluación es más confiable. 80/20 es la convención que funciona en la
mayoría de los casos.

Con 100 filas, un 20% son 20 registros: una evaluación hecha sobre 20 casos se mueve mucho por azar.
Ahí se usa 70/30, para que el conjunto de prueba no quede ridículo. En el otro extremo, con millones
de filas se usa 99/1, porque ese 1% ya son miles de casos y de sobra alcanza.

Lo que **no** se hace nunca es no partir, o partir después de entrenar.

</details>

**Pregunta de interpretación 5.** Suponga que cambia `random_state=42` por `random_state=7` y vuelve
a correr todo el cuaderno. ¿Qué números cambian y cuáles no? ¿Y qué haría si al cambiar la semilla el
R2 pasara de 0.68 a 0.45?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Cambian todos los números concretos.** Otra semilla es otra partición: otras 3.094 filas en
prueba, otro árbol entrenado sobre otras 12.375, y por lo tanto otro R2, otro MAE, otro gap y otro
orden en las importancias del cuarto lugar hacia abajo. Nada de eso está mal: son ejecuciones
igualmente válidas del mismo procedimiento.

**No cambia nada de lo que se concluye.** El orden de magnitud del R2, la forma de la curva de
sobreajuste, que el tipo de suscriptor domina, que la fuga infla el número. Si su conclusión
sobrevive al cambio de semilla, es una conclusión. Si no sobrevive, era ruido con formato de
hallazgo.

**Y por eso el salto de 0.68 a 0.45 sería la señal más útil del cuaderno.** No significaría que la
semilla 42 es "la buena": significaría que el desempeño del modelo depende fuertemente de qué filas
cayeron en prueba, o sea que 3.094 filas no alcanzan para medirlo con estabilidad, o que hay grupos
muy pequeños que a veces caen enteros de un lado. La respuesta correcta no es buscar la semilla que
mejor se ve —eso es hacer trampa con un nombre elegante— sino **reportar el rango**: correr varias
semillas y decir "el R2 de prueba está entre 0.45 y 0.68 según la partición". Fijar la semilla sirve
para que el resultado sea reproducible, no para escoger el resultado.

</details>

---

## 4. El patrón de cinco pasos

**Esto es lo más importante que se lleva del bloque.** Los cinco pasos son idénticos para **todos** los
algoritmos de scikit-learn:

```python
from sklearn.XXX import Modelo             # 1. Importar
modelo = Modelo(parametros)                # 2. Crear
modelo.fit(X_train, y_train)               # 3. Entrenar
predicciones = modelo.predict(X_test)      # 4. Predecir
puntaje = modelo.score(X_test, y_test)     # 5. Evaluar
```

Regresión lineal, random forest, SVM, KNN: cambia el import y el nombre de la clase. Nada más.
Aprender un algoritmo nuevo pasa a ser leer qué parámetros recibe.

### El árbol de decisión: las 20 preguntas

Un árbol de decisión juega a las 20 preguntas. Cada nodo es una pregunta sobre **una** variable, cada
hoja es una predicción, y la pregunta de cada nodo **la elige el algoritmo**, no usted: en cada punto
toma la que mejor separa los datos que le quedan.

`max_depth` es cuántas preguntas seguidas se permiten. Pocas, y el modelo no alcanza a captar el
patrón. Demasiadas, y termina haciendo una pregunta por fila, que es memorizar.

> **Para entender qué está pasando: un modelo es un objeto con estado**
>
> `DecisionTreeRegressor(max_depth=5)` no entrena nada: crea un objeto **vacío**, con las instrucciones
> puestas. `fit(...)` es el método que lo llena: guarda dentro del objeto la regla aprendida. Por eso
> `modelo` significa cosas distintas antes y después de esa línea, aunque se llame igual.
>
> Y por eso los atributos aprendidos terminan en **guion bajo**: `feature_importances_`, `tree_`,
> `n_features_in_`. Es una convención de scikit-learn que significa "esto no existía hasta que
> llamaste a `fit`". La celda de abajo lo provoca.

In [ ]:
# Error 3: usar el modelo antes de entrenarlo
sin_entrenar = DecisionTreeRegressor(max_depth=5, random_state=SEMILLA)

try:
    sin_entrenar.predict(X_test)
except Exception as error:
    print("Tipo de error:", type(error).__name__)
    print("Mensaje:", str(error).split("\n")[0])
    print("-> El objeto existe, pero esta vacio: falta el paso 3, .fit(X_train, y_train).")

In [ ]:
# 1. Importar  (ya esta arriba)
# 2. Crear
modelo = DecisionTreeRegressor(max_depth=5, random_state=SEMILLA)

# 3. Entrenar  -- SOLO con los datos de entrenamiento
modelo.fit(X_train, y_train)

# 4. Predecir  -- sobre los datos de prueba
y_pred = modelo.predict(X_test)

# 5. Evaluar
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("PATRON DE 5 PASOS COMPLETADO")
print()
print(f"MAE : {mae:.2f} m3 por suscriptor")
print(f"R2  : {r2:.3f}")
print()
print(f"El consumo promedio real es {y.mean():.1f} m3, asi que equivocarse por {mae:.2f} m3")
print(f"es como un {100 * mae / y.mean():.0f}% del valor tipico.")

### 4.1 La regla que el modelo dedujo

Prometido en la sección 1: aquí está la regla, escrita. Es un árbol de profundidad 2 —el mismo
algoritmo, más corto, para que quepa— y se lee de arriba abajo como una cadena de preguntas.

Nadie escribió esas preguntas ni esos umbrales. Salieron de los ejemplos.

In [ ]:
arbol_corto = DecisionTreeRegressor(max_depth=2, random_state=SEMILLA)
arbol_corto.fit(X_train, y_train)

print(export_text(arbol_corto, feature_names=list(X.columns)))
print("Cada 'value' es lo que el modelo predice para quien llega hasta esa hoja.")

**Pregunta de interpretación 6.** Lea el árbol impreso arriba como si fuera un `if`: ¿qué predice
para un suscriptor de Público / Oficial con 200 suscriptores en el grupo? ¿Y por qué la **primera**
pregunta que hace el modelo es sobre Público / Oficial y no sobre el mes o el municipio?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**67,27 m3 por suscriptor.** Se sigue la rama: `ESTRATO_Publico / Oficial > 0.50` (o sea, sí lo es),
y luego `No. SUSCRIPTORES ACUEDUCTO > 41.50` (200 lo es), y ahí está la hoja. Ese `> 0.50` es la
manera que tiene el árbol de preguntar "¿esta columna de ceros y unos vale 1?": nunca es exactamente
0,5, así que el umbral parte limpio los dos casos.

**La primera pregunta no la eligió nadie: la eligió el algoritmo,** y eligió la que más reduce el
error al partir en dos. Público / Oficial gana porque separa dos poblaciones que casi no se solapan:
colegios, hospitales y dependencias oficiales consumen por suscriptor en otro orden de magnitud que
una vivienda. Compare las hojas: 12,02 contra 67,27 m3. El mes no aparece porque partir por mes deja
los dos lados casi iguales, y una partición que no separa nada no reduce el error.

Fíjese en lo que acaba de hacer: **le acaba de explicar a alguien qué hace el modelo, leyendo el
modelo.** Eso es lo que hace del árbol el algoritmo del curso. Con una red neuronal esta pregunta no
tendría respuesta.

</details>

### 4.2 El mismo patrón, otra profundidad

Los cinco pasos otra vez, con `max_depth=3`, y con la comparación que importa: la métrica de prueba
al lado de la de entrenamiento. La que se reporta es la primera; la segunda está aquí solo para
poder mirar la distancia entre las dos, que es el tema de la sección 5.

In [ ]:
arbol_3 = DecisionTreeRegressor(max_depth=3, random_state=SEMILLA)
arbol_3.fit(X_train, y_train)

r2_prueba_3 = round(r2_score(y_test, arbol_3.predict(X_test)), 3)

print("R2 de prueba con max_depth=3:", r2_prueba_3)
print("R2 de entrenamiento (que NO se reporta):",
      round(r2_score(y_train, arbol_3.predict(X_train)), 3))

---

## 5. Sobreajuste: verlo, no creerlo

### La regla del gap

```
gap = puntaje de entrenamiento - puntaje de prueba

gap < 0.05        bien
gap 0.05 a 0.10   aceptable
gap > 0.10        SOBREAJUSTE: simplifique el modelo
```

**Sobreajustar es memorizar en vez de generalizar.** El árbol sigue partiendo hasta aislar cada fila de
entrenamiento en su propia hoja: al final tiene una respuesta guardada para cada caso que vio, y
ninguna regla que sirva para uno nuevo.

La tabla de abajo recorre `max_depth` y mira qué le pasa al entrenamiento, a la prueba y al gap. No se
enuncia el sobreajuste: se ve.

**Antes de ejecutar, prediga en voz alta:** ¿qué le va a pasar al puntaje de **entrenamiento** cuando
`max_depth` sea `None`, o sea sin límite de profundidad?

In [ ]:
PROFUNDIDADES = [2, 3, 5, 8, 10, 20, None]


def diagnosticar(gap):
    if gap > 0.10:
        return "SOBREAJUSTE"
    if gap > 0.05:
        return "aceptable"
    return "bien"


resultados = []
for profundidad in PROFUNDIDADES:
    arbol = DecisionTreeRegressor(max_depth=profundidad, random_state=SEMILLA)
    arbol.fit(X_train, y_train)

    r2_tr = r2_score(y_train, arbol.predict(X_train))
    r2_te = r2_score(y_test, arbol.predict(X_test))

    resultados.append({
        "max_depth": str(profundidad),
        "train_r2": round(r2_tr, 3),
        "test_r2": round(r2_te, 3),
        "gap": round(r2_tr - r2_te, 3),
        "mae_test": round(mean_absolute_error(y_test, arbol.predict(X_test)), 2),
        "diagnostico": diagnosticar(r2_tr - r2_te),
    })

tabla = pd.DataFrame(resultados)
print(tabla.to_string(index=False))

### Qué hay que ver en esa tabla

1. **Con `max_depth=None` el entrenamiento llega a 1.000.** ¿Es el mejor modelo? No: significa que el
   árbol aisló cada fila de entrenamiento en su propia hoja. Memorizó. **Un puntaje perfecto en
   entrenamiento es una mala noticia, no una buena.**

2. **La prueba deja de mejorar mucho antes de que el entrenamiento deje de subir.** Entre profundidad
   10 y 20 el entrenamiento sube más de 0.10 y la prueba sube menos de 0.01. Todo ese aprendizaje extra
   fue ruido.

3. **La regla del gap funciona:** cruza 0.10 justo donde la prueba se estanca.

4. **La elección razonable es 8 o 10**, no la que maximiza la prueba. Entre dos modelos que rinden
   parecido en prueba, se elige siempre el más simple: es más fácil de explicar y menos frágil.

El gráfico de abajo dibuja lo mismo. La franja sombreada entre las dos curvas **es** el sobreajuste.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(tabla))
ax.plot(x, tabla["train_r2"], "o-", lw=2.5, label="Entrenamiento (el taller)", color="#2E86AB")
ax.plot(x, tabla["test_r2"], "s-", lw=2.5, label="Prueba (el examen)", color="#C44E52")
ax.fill_between(x, tabla["train_r2"], tabla["test_r2"], alpha=0.15, color="#C44E52")

ax.set_xticks(x)
ax.set_xticklabels(tabla["max_depth"])
ax.set_xlabel("max_depth (cuantas preguntas seguidas puede hacer el arbol)")
ax.set_ylabel("R2")
ax.set_title("El entrenamiento sigue subiendo; la prueba se estanca\n"
             "La franja sombreada es el sobreajuste")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Pregunta de interpretación 7.** Con `max_depth=None` el R2 de entrenamiento da **exactamente
1.000**, no 0.999. Explique con la mecánica del árbol por qué el número es exacto, y por qué eso es
una mala noticia y no un logro.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Es exacto porque el árbol siguió partiendo hasta que en cada hoja quedó un solo grupo de filas
idénticas en `X`.** La predicción de una hoja es el promedio de lo que cayó ahí; si cayó una sola
fila, la predicción **es** su valor, y el error de esa fila es cero. Repetido sobre las 12.375 filas
de entrenamiento, el error total es cero y el R2 es 1 por definición, no por aproximación.

**Y ahí está el problema: eso no es una regla, es una tabla de consulta.** El modelo puede recitar
de memoria las 12.375 respuestas que vio, y no aprendió nada que sirva para la 12.376. Se nota en la
misma fila de la tabla: la prueba se quedó en 0.818, con un gap de 0.182. Todo lo que ganó en
entrenamiento entre profundidad 10 y sin límite —de 0.881 a 1.000— fue memorización pura.

**La regla que hay que llevarse:** un puntaje perfecto en entrenamiento nunca es una buena noticia.
Cuando alguien reporte un R2 de 1.000, o de 0.99, hay exactamente dos explicaciones y las dos son
malas: o midió sobre entrenamiento, o tiene fuga de datos.

</details>

### 5.1 Las profundidades que la tabla no probó

La tabla saltó de 5 a 8. La celda de abajo llena el hueco con las profundidades 6 y 7, que es donde
está la transición de "bien" a "empieza a estirarse".

In [ ]:
gaps_6_7 = []
for profundidad in [6, 7]:
    arbol = DecisionTreeRegressor(max_depth=profundidad, random_state=SEMILLA)
    arbol.fit(X_train, y_train)
    gap = r2_score(y_train, arbol.predict(X_train)) - r2_score(y_test, arbol.predict(X_test))
    gaps_6_7.append(round(gap, 3))
    print(f"max_depth={profundidad}  gap={round(gap, 3)}")

**Pregunta de interpretación 8.** Los gaps dieron 0.032 en profundidad 6 y 0.035 en la 7. Entre la 5
y la 8 el gap sube de 0.030 a 0.046, y entre la 10 y la 20 salta de 0.075 a 0.179. ¿Qué está pasando
en el modelo para que el crecimiento sea tan lento primero y tan brusco después?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Cada nivel nuevo del árbol duplica el número de hojas**, así que la cantidad de reglas crece
exponencialmente mientras la cantidad de datos se queda igual. En los primeros niveles cada hoja
todavía contiene cientos de filas, y la predicción es un promedio sobre muchos casos: eso es una
regla, y generaliza. Hacia la profundidad 10 y más, muchas hojas ya tienen un puñado de filas, y el
promedio de tres filas no describe una población, describe tres filas.

Dicho de otra forma: **el gap no mide la complejidad del modelo, mide cuántos datos le quedan por
hoja al final.** Mientras sobran datos, complejidad extra es aprendizaje; cuando ya no sobran,
complejidad extra es memorización. El punto donde se cruza depende del dataset, y por eso no hay un
`max_depth` universal: hay que mirar la curva en **sus** datos.

Esto también explica por qué en el reto, con 442 filas en vez de 15.469, hasta un árbol de
profundidad 3 sobreajusta. No es que el algoritmo sea peor: es que las hojas se quedan sin datos
muchísimo antes.

</details>

**Pregunta de interpretación 9.** Si tuviera que entregarle **un** modelo a Empocaldas, ¿qué
`max_depth` elegiría? No responda "el que da mejor prueba": justifique con el gap y con lo que le va
a tocar explicar en una reunión.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Profundidad 8 o 10. Las dos tienen la prueba alta (0.78 y 0.81) con un gap todavía chico (0.046 y
0.075), o sea que lo que aprendieron sirve para casos nuevos.

`max_depth=20` y `None` tienen una prueba parecida o un poco mejor, pero con un gap de 0.18: ese
modelo memorizó, y la memorización es frágil. El día que llegue un año nuevo o un municipio con otro
comportamiento, se cae. Además un árbol sin límite tiene miles de hojas y no hay forma de explicárselo
a nadie.

La regla general: **entre dos modelos que rinden parecido en prueba, siempre el más simple.** Es la
navaja de Occam aplicada a modelos, y en un proyecto donde hay que sustentar lo que se hizo, el modelo
que se puede explicar vale más que el que gana por 0.01.

</details>

---

## 6. Qué mide cada métrica, y por qué una sola engaña

Hasta aquí hemos mirado el R2 casi todo el tiempo. Es hora de decir qué mide cada cosa.

| Métrica | Qué mide | Cómo se lee | En qué unidades |
|---------|----------|-------------|-----------------|
| **MAE** | Error absoluto medio: en promedio, por cuánto se equivoca | "Nos equivocamos por 5.6 m3 por suscriptor" | Las mismas de `y`. Es la que entiende el cliente |
| **R2** | Qué proporción de la variación de `y` logra explicar el modelo | "Explica el 68% de la variación" | Sin unidades, de 0 a 1 (puede dar negativo) |

**El R2 tiene un punto de referencia escondido:** vale 0 para un modelo que siempre predice la media,
y 1 para uno perfecto. Por eso un R2 negativo significa que su modelo es peor que no tener modelo. Esa
comparación es tan útil que conviene hacerla explícita, y para eso existe `DummyRegressor`: el modelo
tonto que siempre responde la media, contra el cual hay que ganar.

**Y por qué una sola métrica engaña:** el MAE global promedia por igual a todos los casos, así que un
error grande en un segmento pequeño desaparece dentro del promedio. La segunda celda parte el error por
tipo de suscriptor y muestra exactamente eso.

In [ ]:
tonto = DummyRegressor(strategy="mean")
tonto.fit(X_train, y_train)

print(f"{'modelo':<28} {'R2 prueba':>10} {'MAE prueba':>12}")
print("-" * 52)
print(f"{'Tonto (siempre la media)':<28} "
      f"{r2_score(y_test, tonto.predict(X_test)):>10.3f} "
      f"{mean_absolute_error(y_test, tonto.predict(X_test)):>12.2f}")
print(f"{'Arbol con max_depth=5':<28} "
      f"{r2_score(y_test, modelo.predict(X_test)):>10.3f} "
      f"{mean_absolute_error(y_test, modelo.predict(X_test)):>12.2f}")
print()
print("El tonto da R2 = 0 por construccion: ese es el piso contra el que se compara.")
print("Si un modelo no le gana al tonto, no hay modelo.")

In [ ]:
# El mismo MAE, partido por tipo de suscriptor
errores = pd.DataFrame({
    "real": y_test,
    "prediccion": modelo.predict(X_test),
    "estrato": datos_modelo.loc[y_test.index, "ESTRATO"],
})
errores["error_absoluto"] = (errores["real"] - errores["prediccion"]).abs()

por_estrato = (
    errores.groupby("estrato")
    .agg(casos=("error_absoluto", "size"),
         mae=("error_absoluto", "mean"),
         consumo_promedio=("real", "mean"))
    .round(2)
    .sort_values("mae", ascending=False)
)

print(por_estrato.to_string())
print()
print(f"MAE global: {mean_absolute_error(y_test, modelo.predict(X_test)):.2f} m3")
print("Ese numero unico esconde que en Industrial el error es mas del doble.")

### 6.1 Ganarle al tonto, y por cuánto

El R2 ya dijo que hay modelo. La celda de abajo lo dice otra vez, pero en metros cúbicos, que es la
unidad en la que alguien de Empocaldas puede decidir algo.

In [ ]:
mae_arbol = mean_absolute_error(y_test, modelo.predict(X_test))
mae_tonto = mean_absolute_error(y_test, tonto.predict(X_test))

mae_comparado = [round(mae_arbol, 2), round(mae_tonto, 2)]

print("MAE arbol:", mae_comparado[0], "m3")
print("MAE tonto:", mae_comparado[1], "m3")
print(f"El arbol reduce el error un {100 * (1 - mae_arbol / mae_tonto):.0f}% frente a no tener modelo.")

**Pregunta de interpretación 10.** El árbol se equivoca por 5,56 m3 y el tonto por 11,50: una
reducción del 52%. ¿Es mucho? Y una cosa rara de la tabla de arriba: el tonto dio un R2 de
**-0.000**, negativo. ¿Cómo puede ser negativo si se supone que vale 0?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**"Mucho" no se responde con el porcentaje, se responde con la decisión.** Un 52% de reducción suena
bien, pero lo que hay que preguntar es qué se hace distinto con un error de 5,56 en vez de 11,50 m3
sobre un consumo típico de 17. Si Empocaldas va a dimensionar una tubería, equivocarse por un tercio
del consumo probablemente no alcanza y el modelo no está listo. Si va a **priorizar** qué zonas
estudiar primero, ordenar bien ya sirve, y sí alcanza. El mismo número es suficiente o insuficiente
según la decisión que soporte, y por eso la pregunta 4 de la plantilla del reporte no es decorativa.

**Lo del R2 negativo tiene una explicación exacta.** El R2 vale 0 para un modelo que predice la
media **del conjunto sobre el que se evalúa**. El `DummyRegressor` aprendió la media del
entrenamiento y la aplica a prueba, y esas dos medias no son idénticas: la de prueba difiere un poco
por la partición aleatoria. Predecir la media de otro conjunto es un pelo peor que predecir la
propia, y de ahí el signo. El -0.000 es esa diferencia diminuta, no un error.

Lo importante es que **el piso existe y se puede calcular**. Un R2 claramente negativo —un -0.4, no
un -0.000— significa que el modelo es peor que responder siempre el promedio, y eso pasa más de lo
que la gente cree cuando se evalúa por primera vez sobre prueba de verdad.

</details>

**Pregunta de interpretación 11.** ¿Cuál es un buen R2? ¿0.68 es bueno? ¿Y 0.98?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No hay tabla universal, y esa es la respuesta: depende del dominio. En datos sociales o médicos, un
0.3 ya es informativo, porque el comportamiento humano tiene un ruido enorme. En datos operativos o
físicos se espera 0.7 o más.

El 0.68 de hoy es razonable para el problema, y viene acompañado de un MAE de 5.56 m3 sobre un consumo
típico de 17 m3: un error de casi un tercio. **No es un gran modelo, y hay que decirlo.** Un modelo
mediocre bien reportado vale más que uno excelente inventado.

Un 0.98 en un problema del mundo real debería hacer sospechar antes que celebrar: casi siempre es fuga
de datos. La sección que sigue lo demuestra.

</details>

### 6.2 La fuga de datos, ejecutada

Ahora sí, la versión prohibida: metemos `CONSUMO M3 ACUEDUCTO` dentro de `X`. Recuerde que
`PROMEDIO ≈ CONSUMO M3 / SUSCRIPTORES`, así que le estamos entregando la respuesta al modelo.

**Prediga antes de ejecutar:** ¿qué R2 cree que va a salir?

In [ ]:
datos_fuga = modelo_df[COLUMNAS_ENTRADA + ["CONSUMO M3 ACUEDUCTO", COLUMNA_OBJETIVO]].dropna()

X_fuga = pd.get_dummies(datos_fuga[COLUMNAS_ENTRADA + ["CONSUMO M3 ACUEDUCTO"]],
                        columns=["ESTRATO", "MUNICIPIO"], drop_first=True)
y_fuga = datos_fuga[COLUMNA_OBJETIVO]

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fuga, y_fuga, test_size=0.2, random_state=SEMILLA)

print(f"{'max_depth':>10} | {'honesto':>8} | {'CON FUGA':>9}")
print("-" * 33)
for profundidad in [5, 10, 20]:
    honesto = DecisionTreeRegressor(max_depth=profundidad, random_state=SEMILLA)
    honesto.fit(X_train, y_train)
    tramposo = DecisionTreeRegressor(max_depth=profundidad, random_state=SEMILLA)
    tramposo.fit(Xf_train, yf_train)
    print(f"{profundidad:>10} | {r2_score(y_test, honesto.predict(X_test)):>8.3f} | "
          f"{r2_score(yf_test, tramposo.predict(Xf_test)):>9.3f}")

print()
print("El modelo con fuga se ve mucho mejor. Y es inutil: el dia que llegue un barrio")
print("nuevo no va a existir el consumo total, porque eso es lo que se queria predecir.")
print()
print("REGLA: en X solo va informacion que tendria ANTES de conocer y.")

**Pregunta de interpretación 12.** El modelo con fuga llega a 0.981 en profundidad 20, contra 0.809
del honesto. Y el gap del tramposo, si lo calculara, también se vería sano. Entonces: **si nadie le
dice cuál es la columna prohibida, ¿cómo detectaría una fuga en su propio dataset?** Dé al menos dos
señales.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Ese es exactamente el punto peligroso: **la fuga no se detecta con las herramientas de esta clase.**
La partición está bien hecha, el gap se ve bien, la métrica es de prueba. Todo el aparato de
validación dice que el modelo es excelente, porque la trampa entró antes de que empezara a medir.
Por eso la fuga es el error más caro del proyecto: no falla, brilla.

**Señal 1: el resultado es demasiado bueno para el problema.** Predecir comportamiento humano o
consumo con un R2 de 0.98 no pasa. Cuando un número le dé ganas de celebrar, el reflejo correcto es
ir a buscar de dónde salió, no ponerlo en la primera diapositiva.

**Señal 2: una sola variable concentra casi toda la importancia.** Si `feature_importances_` le
entrega 0.95 a una columna, esa columna probablemente **es** la respuesta con otro nombre. Vale la
pena mirar también su correlación con `y`: un 0.99 no es una variable predictiva, es un duplicado.

**Señal 3, y es la única que de verdad funciona: la línea de tiempo.** Se toma cada columna de `X` y
se pregunta *¿en qué momento se genera este dato en el mundo real, antes o después de `y`?* Es una
pregunta sobre el proceso del negocio, no sobre el DataFrame, y por eso hay que hablar con quien
conoce el dominio. Ningún chequeo estadístico la reemplaza.

</details>

---

## 7. Interpretar el modelo sin inventar causalidad

Un árbol entrenado expone `feature_importances_`: un número por cada columna de `X` que mide cuánto
contribuyó esa variable a reducir el error al partir los datos. Suman 1.

Esto es lo que hace que el árbol sea el algoritmo del curso: **se puede explicar**. Puede mostrarle
esto a alguien de Empocaldas y va a entender por qué el modelo predice lo que predice.

In [ ]:
importancias = (
    pd.Series(modelo.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)

print(importancias.round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
importancias.sort_values().plot(kind="barh", ax=ax, color="#2E86AB")
ax.set_xlabel("Importancia (suman 1 entre todas las variables de X)")
ax.set_ylabel("Variable de X")
ax.set_title("El tipo de suscriptor manda sobre todo lo demas\n"
             "Top 10 del arbol con max_depth=5")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

**Pregunta de interpretación 13.** En el top 10 no aparece `MUNICIPIO_MANIZALES`, y `mes_num` está
tan abajo que no entra. ¿Significa eso que el municipio de Manizales y el mes **no informan nada**
sobre el consumo? Cuidado: las dos ausencias no tienen la misma explicación.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No, y por dos motivos distintos, que es justo lo que hace difícil leer este gráfico.**

`MUNICIPIO_MANIZALES` no aparece por una razón mecánica: `drop_first=True` elimina una de las 24
columnas de municipio, y la eliminada no puede tener importancia porque **no existe** en `X`. Su
información no se perdió —está implícita en que todas las demás valgan cero— pero el crédito se lo
llevan las otras. Importancia cero por construcción, no por irrelevancia.

`mes_num` sí existe en `X` y sí tiene importancia casi cero, y eso sí es un hallazgo: **la
estacionalidad del consumo de agua en Caldas es débil.** No es un defecto del modelo, es una
propiedad de los datos, y merece una frase en el informe tanto como las variables que sí dominaron.

**La regla general que hay detrás:** importancia baja significa "el árbol no la necesitó para
partir", que no es lo mismo que "no sirve". Puede ser que otra variable correlacionada se llevara
todo el crédito, que la variable esté implícita en la codificación, o que efectivamente no aporte.
Distinguir los tres casos requiere mirar los datos, no el gráfico.

</details>

### La lectura de negocio

Traducido a algo que se pueda decir en una reunión:

- **El tipo de suscriptor manda sobre todo lo demás.** Un colegio o un hospital consume por suscriptor
  en otro orden de magnitud que una vivienda, y el modelo lo detecta antes que cualquier otra cosa.
- **El tamaño del grupo importa.** El número de suscriptores es la segunda variable.
- **El municipio importa poco, salvo casos puntuales.** La Dorada y Guarinocito aparecen porque son las
  zonas cálidas de Caldas, donde se consume más agua.
- **El mes casi no aparece.** La estacionalidad del consumo de agua en Caldas es débil, y eso también
  es un hallazgo.

### Lo que esto NO dice, y es la parte que se pierde en las sustentaciones

**Un modelo no explica causalidad.** Es exactamente lo mismo que se dijo en la clase 5 sobre la
correlación, y aquí vuelve con otro disfraz: allá era un coeficiente alto entre dos columnas, aquí es
una barra alta en un gráfico de importancias. En los dos casos lo que hay es **asociación medida en
unos datos**, no un mecanismo.

Decir "el estrato Público / Oficial **causa** un consumo alto" es tan falso como decirlo desde una
correlación. Lo que se puede afirmar es: *"en estos datos, el tipo de suscriptor es la variable que más
usó el modelo para separar consumos altos de bajos"*. Esa frase se sostiene; la otra no.

Tres advertencias más, que no son opcionales:

1. **Con variables correlacionadas, el árbol elige una y le da todo el crédito.** La otra aparece con
   importancia casi cero aunque contenga la misma información. Importancia cero no significa "no
   sirve".
2. **La importancia es una propiedad del modelo entrenado, no del mundo.** Cambie `max_depth` y cambia
   el ranking. La sección 7.1, justo abajo, lo comprueba entrenando el mismo algoritmo con otra
   profundidad y comparando los dos rankings.
3. **Si el modelo generaliza mal, sus importancias describen un modelo malo.** Se reportan junto al gap,
   nunca sueltas.

### 7.1 ¿El ranking es estable?

La advertencia 3 dice que la importancia es una propiedad del modelo entrenado, no del mundo. La
celda de abajo lo pone a prueba de la única forma seria: entrena el mismo algoritmo con otra
profundidad y compara los dos rankings, uno debajo del otro.

In [ ]:
arbol_10 = DecisionTreeRegressor(max_depth=10, random_state=SEMILLA)
arbol_10.fit(X_train, y_train)

importancias_10 = pd.Series(arbol_10.feature_importances_, index=X.columns)
top_5_profundidad_10 = importancias_10.sort_values(ascending=False).head(5).index.tolist()

print("Profundidad 10:", top_5_profundidad_10)
print("Profundidad  5:", importancias.head(5).index.tolist())

**Pregunta de interpretación 14.** ¿Cambió el ranking al aumentar la profundidad? ¿Cuánta confianza
le daría entonces a `feature_importances_` cuando lo ponga en su informe del Momento 3?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Las dos primeras variables se sostienen (el tipo de suscriptor y el número de suscriptores), pero de la
tercera en adelante el orden se mueve: con profundidad 10 entran variables que con profundidad 5 no
aparecían.

La lectura correcta: **las importancias son estables en lo grueso y frágiles en el detalle.** Sirven
para decir "el tipo de suscriptor domina", que se sostiene con cualquier configuración razonable. No
sirven para decir "la quinta variable más importante es X", que cambia si alguien mueve un parámetro.

En el informe eso se traduce en cómo se escribe la frase: se reporta el orden de magnitud y las dos o
tres variables que dominan, no un ranking completo presentado como si fuera una medición del mundo.

</details>

---

## Cierre del demo

### Lo que hizo hoy

| Herramienta | Para qué |
|-------------|----------|
| `train_test_split` | Apartar el 20% para el examen |
| `DecisionTreeRegressor` | Predecir un número |
| `.fit()` / `.predict()` | Los pasos 3 y 4 del patrón universal |
| `mean_absolute_error`, `r2_score` | Evaluar una regresión, en dos escalas distintas |
| `DummyRegressor` | El piso contra el que se compara cualquier modelo |
| `.feature_importances_` | Entender qué usó el modelo |
| `pd.get_dummies` | Convertir categorías en columnas de 0 y 1 |
| `export_text` | Leer la regla que el modelo dedujo |

### Lo que hay que recordar aunque se olvide el código

1. Entrenar es **deducir una regla a partir de ejemplos**, no programarla.
2. La métrica que cuenta es la de **prueba**, nunca la de entrenamiento.
3. `gap > 0.10` es sobreajuste, y un entrenamiento perfecto es mala noticia.
4. En `X` solo va lo que tendría **antes** de conocer `y`.
5. Una sola métrica esconde cosas: reporte al menos dos, y compare contra el modelo tonto.
6. Importancia no es causalidad. Igual que la correlación de la clase 5.
7. Un modelo mediocre bien reportado vale más que uno excelente inventado.

### La plantilla del reporte de modelo

Las cuatro preguntas que la capa de ML de su proyecto final tiene que responder:

1. **¿Qué predijo?** La variable objetivo, y por qué le importa a alguien.
2. **¿Es regresión o clasificación?** Y por qué esa y no la otra.
3. **¿Cómo evaluó?** Métrica sobre **prueba**, más el gap contra entrenamiento.
4. **¿Qué decisión soporta la predicción?** Si no soporta ninguna, el modelo sobra.

Aplicado a lo de hoy:

> Predecimos el consumo promedio por suscriptor de acueducto (m3/mes) para que Empocaldas pueda
> dimensionar infraestructura en zonas sin historia. Es regresión porque la variable objetivo es un
> número continuo. Un árbol con `max_depth=8` alcanza un R2 de 0.778 sobre prueba con un gap de 0.046,
> o sea que generaliza aceptablemente. El error típico es de 4.37 m3 por suscriptor, una cuarta parte
> del consumo promedio, y es mayor en Industrial y Público / Oficial que en los estratos residenciales.
> El tipo de suscriptor concentra más de la mitad de la capacidad predictiva, lo que sugiere segmentar
> la planeación por tipo de suscriptor antes que por municipio.

### Ahora al reto

En `../reto/` va a construir la curva de sobreajuste completa sobre datos **que no aparecieron aquí**:
`load_diabetes` de scikit-learn, que no requiere ningún archivo. Otro dominio, otra escala, la misma
técnica.

Aviso por adelantado: **en ese dataset todos los modelos sobreajustan.** Es esperado, y la respuesta
correcta es reportarlo, no esconderlo.